In [1]:
import os
import pandas as pd
import sqlite3


while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print("cwd:", os.getcwd())

master = pd.read_csv('data/processed/aqi_weather_master.csv', parse_dates=['date'])
print(master.shape)

cwd: C:\Users\91935\PycharmProjects\AQi project
(15283, 19)


In [2]:
os.makedirs('db', exist_ok=True)
conn = sqlite3.connect('db/aqi.db')

In [3]:
master.to_sql('aqi_weather', conn, if_exists='replace', index=False)

print(pd.read_sql("SELECT COUNT(*) AS n_rows FROM aqi_weather", conn))
print(pd.read_sql("SELECT date FROM aqi_weather LIMIT 3", conn))

   n_rows
0   15283
                  date
0  2015-01-01 00:00:00
1  2015-01-02 00:00:00
2  2015-01-03 00:00:00


In [9]:
q1 = """
WITH monthly AS (
    SELECT
        city,
        strftime('%Y-%m', date)  AS year_month,
        ROUND(AVG(AQI), 1)       AS avg_aqi,
        COUNT(AQI)               AS days_with_aqi
    FROM aqi_weather
    WHERE AQI IS NOT NULL
    GROUP BY city, strftime('%Y-%m', date)
    HAVING COUNT(AQI) >= 20
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY city ORDER BY avg_aqi DESC) AS rnk
    FROM monthly
)
SELECT city, rnk AS rank, year_month, avg_aqi, days_with_aqi
FROM ranked
WHERE rnk <= 5
ORDER BY city, rnk;
"""
result1 = pd.read_sql(q1, conn)


def run_and_save(sql_text, filename, comment):
    with open(f'sql/{filename}', 'w', encoding='utf-8') as f:
        f.write(f"-- {comment}\n{sql_text.strip()}\n")
    return pd.read_sql(sql_text, conn)
result1 = run_and_save(q1, '01_worst_months.sql',
                       'Worst 5 months per city by average AQI (months with at least 20 days of AQI data)')
print(result1.to_string(index=False))

     city  rank year_month  avg_aqi  days_with_aqi
Bengaluru     1    2015-07    160.5             31
Bengaluru     2    2016-03    145.6             31
Bengaluru     3    2016-02    143.9             29
Bengaluru     4    2015-08    134.9             31
Bengaluru     5    2016-01    134.3             31
  Chennai     1    2016-09    184.9             30
  Chennai     2    2015-05    176.1             31
  Chennai     3    2015-04    175.9             29
  Chennai     4    2025-12    166.5             31
  Chennai     5    2015-07    164.1             31
    Delhi     1    2016-11    474.8             30
    Delhi     2    2016-01    441.9             31
    Delhi     3    2016-10    411.1             31
    Delhi     4    2016-12    409.3             31
    Delhi     5    2017-11    406.7             30
  Kolkata     1    2019-01    360.8             31
  Kolkata     2    2018-12    338.3             31
  Kolkata     3    2018-11    283.7             30
  Kolkata     4    2019-02    2

In [6]:
q2 = """
SELECT
    city,
    strftime('%Y', date)                         AS year,
    COUNT(AQI)                                   AS days_with_aqi,
    SUM(CASE WHEN AQI > 300 THEN 1 ELSE 0 END)   AS days_above_300,
    ROUND(100.0 * SUM(CASE WHEN AQI > 300 THEN 1 ELSE 0 END) / COUNT(AQI), 1) AS pct_above_300
FROM aqi_weather
WHERE AQI IS NOT NULL
GROUP BY city, strftime('%Y', date)
ORDER BY city, year;
"""
result2 = pd.read_sql(q2, conn)

result2 = run_and_save(q2, '02_days_above_300.sql',
                       'Days with AQI above 300 per city per year, with percentage of days with data')
print(result2.to_string(index=False))

     city year  days_with_aqi  days_above_300  pct_above_300
Bengaluru 2015            286               1            0.3
Bengaluru 2016            351               3            0.9
Bengaluru 2017            364               0            0.0
Bengaluru 2018            361               1            0.3
Bengaluru 2019            365               0            0.0
Bengaluru 2020            183               0            0.0
Bengaluru 2025            153               0            0.0
Bengaluru 2026            212               0            0.0
  Chennai 2015            276               6            2.2
  Chennai 2016            334               5            1.5
  Chennai 2017            361               5            1.4
  Chennai 2018            365               0            0.0
  Chennai 2019            365               1            0.3
  Chennai 2020            183               0            0.0
  Chennai 2025            153               8            5.2
  Chennai 2026          

In [7]:
def run_and_save(sql_text, filename, comment):
    with open(f'sql/{filename}', 'w', encoding='utf-8') as f:
        f.write(f"-- {comment}\n{sql_text.strip()}\n")
    return pd.read_sql(sql_text, conn)

q3 = """
WITH seasonal AS (
    SELECT
        city,
        CASE
            WHEN CAST(strftime('%m', date) AS INTEGER) IN (11, 12, 1, 2) THEN 'winter'
            WHEN CAST(strftime('%m', date) AS INTEGER) IN (3, 4, 5, 6)   THEN 'summer'
        END AS season_group,
        AQI
    FROM aqi_weather
    WHERE AQI IS NOT NULL
)
SELECT
    city,
    ROUND(AVG(CASE WHEN season_group = 'winter' THEN AQI END), 1) AS winter_avg,
    ROUND(AVG(CASE WHEN season_group = 'summer' THEN AQI END), 1) AS summer_avg,
    ROUND(AVG(CASE WHEN season_group = 'winter' THEN AQI END)
        / AVG(CASE WHEN season_group = 'summer' THEN AQI END), 2) AS winter_to_summer_ratio
FROM seasonal
WHERE season_group IS NOT NULL
GROUP BY city
ORDER BY winter_to_summer_ratio DESC;
"""

result3 = run_and_save(q3, '03_winter_summer_ratio.sql',
                       'Average AQI in winter (Nov-Feb) vs summer (Mar-Jun) per city, and their ratio')
print(result3.to_string(index=False))

     city  winter_avg  summer_avg  winter_to_summer_ratio
  Kolkata       231.1        79.1                    2.92
    Patna       324.2       162.2                    2.00
  Lucknow       309.5       175.4                    1.76
   Mumbai       150.8        89.5                    1.68
    Delhi       353.0       221.6                    1.59
  Chennai       122.9       103.9                    1.18
Bengaluru       104.9        89.1                    1.18


In [10]:
q4 = """
SELECT
    city,
    CASE
        WHEN AQI <= 50  THEN '1 Good'
        WHEN AQI <= 100 THEN '2 Satisfactory'
        WHEN AQI <= 200 THEN '3 Moderate'
        WHEN AQI <= 300 THEN '4 Poor'
        WHEN AQI <= 400 THEN '5 Very Poor'
        ELSE                 '6 Severe'
    END AS aqi_category,
    COUNT(*) AS days,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY city), 1) AS pct_of_city_days
FROM aqi_weather
WHERE AQI IS NOT NULL
GROUP BY city, aqi_category
ORDER BY city, aqi_category;
"""
result4 = run_and_save(q4, '04_days_by_category.sql',
                       'Number and share of days per CPCB AQI category, per city')
print(result4.to_string(index=False))

     city   aqi_category  days  pct_of_city_days
Bengaluru         1 Good   220               9.7
Bengaluru 2 Satisfactory  1292              56.8
Bengaluru     3 Moderate   720              31.6
Bengaluru         4 Poor    38               1.7
Bengaluru    5 Very Poor     5               0.2
  Chennai         1 Good    80               3.6
  Chennai 2 Satisfactory  1147              51.0
  Chennai     3 Moderate   875              38.9
  Chennai         4 Poor   120               5.3
  Chennai    5 Very Poor    17               0.8
  Chennai       6 Severe    11               0.5
    Delhi         1 Good    31               1.3
    Delhi 2 Satisfactory   217               9.1
    Delhi     3 Moderate   681              28.7
    Delhi         4 Poor   580              24.4
    Delhi    5 Very Poor   600              25.3
    Delhi       6 Severe   265              11.2
  Kolkata         1 Good   196              17.5
  Kolkata 2 Satisfactory   447              39.9
  Kolkata     3 Mode